# Search lab: from UCS to A*

**Digital Innovation and Product Development**

This is the rail network from the recording. A courier travels from **Central (S)** to
**Greenpark (G)**, and the numbers are journey times in minutes.

In the recording you saw that uniform-cost search finds the quickest route but expands
seven stations to do it, while A* finds the same route after expanding only four.

In this lab you'll write A* yourself and check that it does the same.

1. Run uniform-cost search, which is written for you, and check the numbers against the slides.
2. Write A* by changing UCS in two places.
3. Check that A* returns the same cost as UCS.
4. Break the heuristic on purpose and watch A* return a worse route.

**Working rules.** Work in threes, mixing pathways where you can. You may use AI to write
the code. You may not use it to decide whether the code is right: Part 3 is how you check that.

In [2]:
# The network from the recording.
# Each station lists its neighbours and the journey time in minutes.

GRAPH = {
    "S": {"W": 5, "V": 5, "A": 7, "B": 8},
    "W": {"S": 5, "U": 5},
    "V": {"S": 5, "U": 5},
    "U": {"W": 5, "V": 5},
    "A": {"S": 7, "C": 7},
    "B": {"S": 8, "C": 8},
    "C": {"A": 7, "B": 8, "G": 5},
    "G": {"C": 5},
}

NAMES = {"S": "Central", "W": "Westgate", "V": "Vale End", "U": "Upton",
         "A": "Ashby", "B": "Bellway", "C": "Crossway", "G": "Greenpark"}

# The heuristic: straight-line time from each station to Greenpark.
H = {"S": 14, "W": 17, "V": 17, "U": 19, "A": 10, "B": 10, "C": 4, "G": 0}

def show(label, path, cost, expanded):
    print(f"{label}")
    print(f"  route    : {' -> '.join(path)}  ({cost} minutes)")
    print(f"  expanded : {', '.join(expanded)}  ({len(expanded)} stations)")

## Part 1: uniform-cost search

Read this before running it. It is the loop from the recording:

- the frontier holds `(g, station, path)`, where `g` is the cost so far
- it is sorted by `g`, so the cheapest path comes out first
- the goal test happens when a station is **taken out** of the frontier, not when it is generated
- if a cheaper path to a station turns up, the old frontier entry is replaced

In [3]:
def ucs(graph, start, goal):
    """Uniform-cost search. Returns (path, cost, expanded)."""
    frontier = [(0, start, [start])]       # (cost so far, station, path taken)
    best = {start: 0}                      # cheapest cost found to each station
    expanded = []                          # stations taken out of the frontier

    while frontier:
        frontier.sort(key=lambda item: item[0])          # cheapest first
        g, node, path = frontier.pop(0)

        if node == goal:                                  # goal test on removal
            return path, g, expanded
        expanded.append(node)

        for nxt, cost in sorted(graph[node].items()):
            new_g = g + cost
            if new_g < best.get(nxt, float("inf")):       # a cheaper way to get there
                best[nxt] = new_g
                frontier = [item for item in frontier if item[1] != nxt]
                frontier.append((new_g, nxt, path + [nxt]))

    return None, None, expanded

path, cost, expanded = ucs(GRAPH, "S", "G")
show("UCS", path, cost, expanded)

UCS
  route    : S -> A -> C -> G  (19 minutes)
  expanded : S, V, W, A, B, U, C  (7 stations)


## Part 2: write A*

Copy the `ucs` function into the cell below and change **two** things.

1. **Sort by f, not g.** Where UCS sorts by cost so far, A* sorts by `f = g + h[station]`.
   Keep `g` in the frontier as well, because you still need it to score the neighbours.
2. **Score neighbours with f when you add them,** but keep comparing `new_g` against `best`.
   The record of what is cheapest is still about real cost, not estimates.

Leave everything else alone. In particular, keep the goal test where it is: when a station
comes out of the frontier. Checking as soon as you generate it breaks A*, because seeing the
goal isn't the same as having found the cheapest route to it.

Hint: UCS keeps `(g, station, path)` in the frontier. A* needs f as well, so keep
`(f, g, station, path)`. With f first, sorting the list puts the lowest f at the front.

In [4]:
def astar(graph, start, goal, h):
    """A* search. Returns (path, cost, expanded)."""
    frontier = [(h[start], 0, start, [start])]      # (f = g + h, g, station, path)
    best = {start: 0}
    expanded = []

    while frontier:
        frontier.sort(key=lambda item: item[0])      # lowest f first  <-- change 1
        f, g, node, path = frontier.pop(0)

        if node == goal:                             # unchanged: goal test on removal
            return path, g, expanded
        expanded.append(node)

        for nxt, cost in sorted(graph[node].items()):
            new_g = g + cost                         # compare real cost, not f
            if new_g < best.get(nxt, float("inf")):
                best[nxt] = new_g
                frontier = [item for item in frontier if item[2] != nxt]
                frontier.append((new_g + h[nxt], new_g, nxt, path + [nxt]))   # <-- change 2

    return None, None, expanded

## Part 3: check it before you trust it

Your A* runs. That doesn't mean it's right, and getting 19 minutes on this one route doesn't
prove much either.

Design a test of your own, using this fact:

> **With an admissible heuristic, A\* must return the same cost as UCS.** Anything else is a bug.

Write it and run it. Then ask yourselves: if my A* had a bug, would this test have told me?

In [5]:
# Testing from Central alone won't tell you much. A version of A* that checks for the
# goal when a station is generated, rather than when it comes out of the frontier, is
# broken, but it still returns 19 minutes here. So we run from several stations: a bug
# that hides on one route tends to show up on another.
#
# Notice what we compare. The costs have to match, because both algorithms are optimal.
# The routes don't: when two routes tie on cost, UCS and A* can pick different ones, and
# a test that compares routes would flag that as a failure when nothing is wrong.

def check(starts, goal="G"):
    all_ok = True
    for start in starts:
        _, c_ucs, e_ucs = ucs(GRAPH, start, goal)
        _, c_a,   e_a   = astar(GRAPH, start, goal, H)
        ok = (c_ucs == c_a)
        all_ok = all_ok and ok
        print(f"{NAMES[start]:>9} -> {NAMES[goal]:<9} | "
              f"UCS {c_ucs:>2} min, {len(e_ucs)} expanded | "
              f"A* {c_a:>2} min, {len(e_a)} expanded | "
              f"{'ok' if ok else 'MISMATCH'}")
    # One clear verdict at the end, so you can't read four "ok"s and miss the fifth.
    print("\nAll costs agree." if all_ok else "\nA* is wrong. Fix it before going on.")

# The expansion counts aren't part of the correctness check, but they cost nothing to
# collect and they're the reason we wrote A* at all: 4 stations against 7 from Central.
# Watch the Upton row, though. A* expands all seven there, exactly like UCS, and it is
# still correct. Upton sits in the far corner, so every station is on a plausible route
# and the heuristic has nothing to rule out.
check(["S", "W", "V", "B", "U"])

  Central -> Greenpark | UCS 19 min, 7 expanded | A* 19 min, 4 expanded | ok
 Westgate -> Greenpark | UCS 24 min, 7 expanded | A* 24 min, 6 expanded | ok
 Vale End -> Greenpark | UCS 24 min, 7 expanded | A* 24 min, 6 expanded | ok
  Bellway -> Greenpark | UCS 13 min, 3 expanded | A* 13 min, 2 expanded | ok
    Upton -> Greenpark | UCS 29 min, 7 expanded | A* 29 min, 7 expanded | ok

All costs agree.


## Part 4: break the heuristic

Our heuristic is admissible: it never overestimates the true cost to Greenpark.

Below, the estimate for Ashby is raised to 18 minutes, when the real cost from Ashby is 12.
Run it and see what A* returns.

Then answer, in one or two sentences: **how would anyone using this system know the route
was wrong?**

In [6]:
H_BAD = dict(H)
H_BAD["A"] = 18          # true cost from Ashby is 12, so this overestimates

p_bad, c_bad, e_bad = astar(GRAPH, "S", "G", H_BAD)
show("A* with an overestimating heuristic", p_bad, c_bad, e_bad)
print(f"\nOptimal is 19 minutes, so this route is {c_bad - 19} minutes worse.")

A* with an overestimating heuristic
  route    : S -> B -> C -> G  (21 minutes)
  expanded : S, B, C  (3 stations)

Optimal is 19 minutes, so this route is 2 minutes worse.


**They wouldn't.** A* returns a complete, sensible-looking route with a cost attached, and
nothing in the output says the estimate was wrong or that a better route existed. You'd only
find out by running something you already trust, such as UCS, and comparing the two.